# NLI exploration for main-claim sentences

This notebook uses the Hugging Face model `cross-encoder/nli-deberta-v3-base` as a sentence-level cross-encoder.
For each article sentence, we treat the sentence as the premise and score it against the hypothesis:

`This sentence states the author's main claim.`

A higher entailment probability means the model thinks that sentence is more likely to express the article's central claim.


In [49]:
from pathlib import Path
import re
import sys
from textwrap import shorten

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from IPython.display import Markdown, display

PROJECT_ROOT = next(
    (path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / 'backend').exists()),
    Path.cwd().resolve(),
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from backend.data_import import load_and_clean_guardian_years

MODEL_NAME = 'cross-encoder/nli-deberta-v3-base'
HYPOTHESIS = "This sentence states the author's main opinion."
RAW_DATA_DIR = PROJECT_ROOT / 'backend' / 'data' / 'raw' / 'guardian_by_year'
YEARS = list(range(2015, 2025))
MIN_BODY_TEXT_CHARS = 1000

pd.set_option('display.max_colwidth', 160)
DEVICE = torch.device('mps')
DEVICE


device(type='mps')

In [19]:
def normalize_whitespace(text):
    return re.sub(r'\s+', ' ', str(text)).strip()


def split_into_sentences(text, min_chars=35, min_words=5):
    normalized = normalize_whitespace(text)
    if not normalized:
        return []

    parts = re.split(r'(?<=[.!?])\s+', normalized)

    sentences = []
    seen = set()
    for part in parts:
        sentence = normalize_whitespace(part)
        if len(sentence) < min_chars or len(sentence.split()) < min_words:
            continue
        if sentence not in seen:
            sentences.append(sentence)
            seen.add(sentence)
    return sentences


def pick_article(df, article_id=None, title_query=None, article_index=0):
    matches = df
    if article_id:
        matches = matches.loc[matches['id'].astype('string').str.strip() == str(article_id).strip()]
    elif title_query:
        matches = matches.loc[matches['title'].astype('string').str.contains(title_query, case=False, na=False)]

    matches = matches.reset_index(drop=True)
    if matches.empty:
        raise ValueError('No matching article found. Try a different ARTICLE_ID or TITLE_QUERY.')
    if not 0 <= int(article_index) < len(matches):
        raise IndexError(f'article_index {article_index} is out of range for {len(matches)} matching articles.')
    return matches, matches.iloc[int(article_index)]


def preview_article(article_row, body_width=1400):
    date_value = article_row.get('date', '')
    if pd.notna(date_value):
        try:
            date_value = pd.to_datetime(date_value).strftime('%Y-%m-%d')
        except Exception:
            date_value = str(date_value)

    display(Markdown(f"## {article_row['title']}"))
    print(f"id: {article_row['id']}")
    print(f"date: {date_value}")
    print(f"author(s): {article_row.get('author_display', '')}")
    print()
    print('Summary:')
    print(shorten(normalize_whitespace(article_row.get('summary', '')), width=320, placeholder=' ...'))
    print()
    print('Body preview:')
    print(shorten(normalize_whitespace(article_row.get('body_text', '')), width=body_width, placeholder=' ...'))


def model_device(model):
    return next(model.parameters()).device


def load_nli_model(model_name=MODEL_NAME, device=DEVICE):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    model.to(device)
    model.eval()

    id2label = {int(idx): str(label).lower() for idx, label in model.config.id2label.items()}

    def label_index(fragment):
        for idx, label in id2label.items():
            if fragment in label:
                return idx
        raise KeyError(f'Could not find a label containing {fragment!r} in {id2label}')

    label_map = {
        'contradiction': label_index('contrad'),
        'neutral': label_index('neutral'),
        'entailment': label_index('entail'),
    }
    return tokenizer, model, id2label, label_map


def score_sentence_pairs(premises, hypothesis, tokenizer, model, label_map, batch_size=16, max_length=512):
    if not premises:
        return pd.DataFrame(columns=['entailment_prob', 'neutral_prob', 'contradiction_prob'])

    all_probs = []
    target_device = model_device(model)
    with torch.inference_mode():
        for start in range(0, len(premises), batch_size):
            batch = premises[start : start + batch_size]
            encoded = tokenizer(
                batch,
                [hypothesis] * len(batch),
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt',
            )
            encoded = {key: value.to(target_device) for key, value in encoded.items()}
            probs = torch.softmax(model(**encoded).logits, dim=-1).cpu().numpy()
            all_probs.append(probs)

    probs = np.vstack(all_probs)
    return pd.DataFrame(
        {
            'entailment_prob': probs[:, label_map['entailment']],
            'neutral_prob': probs[:, label_map['neutral']],
            'contradiction_prob': probs[:, label_map['contradiction']],
        }
    )


def score_article_sentences(
    article_row,
    tokenizer,
    model,
    label_map,
    hypothesis=HYPOTHESIS,
    min_sentence_chars=35,
    min_sentence_words=5,
    batch_size=16,
):
    sentences = split_into_sentences(
        article_row.get('body_text', ''),
        min_chars=min_sentence_chars,
        min_words=min_sentence_words,
    )
    if not sentences:
        return pd.DataFrame(
            columns=[
                'sentence_idx',
                'sentence',
                'entailment_prob',
                'neutral_prob',
                'contradiction_prob',
                'claim_margin',
                'hypothesis',
                'article_id',
                'title',
            ]
        )

    probs = score_sentence_pairs(
        premises=sentences,
        hypothesis=hypothesis,
        tokenizer=tokenizer,
        model=model,
        label_map=label_map,
        batch_size=batch_size,
    )

    scored = pd.DataFrame(
        {
            'sentence_idx': range(len(sentences)),
            'sentence': sentences,
        }
    ).join(probs)

    scored['claim_margin'] = scored['entailment_prob'] - scored['contradiction_prob']
    scored['hypothesis'] = hypothesis
    scored['article_id'] = article_row['id']
    scored['title'] = article_row['title']

    return scored.sort_values(
        ['entailment_prob', 'claim_margin', 'sentence_idx'],
        ascending=[False, False, True],
    ).reset_index(drop=True)


def collect_top_claim_sentence_per_article(
    df,
    tokenizer,
    model,
    label_map,
    hypothesis=HYPOTHESIS,
    sample_size=5,
    random_state=7,
    min_sentence_chars=35,
    min_sentence_words=5,
    batch_size=16,
):
    if df.empty:
        return pd.DataFrame()

    sampled = df.sample(n=min(sample_size, len(df)), random_state=random_state)
    rows = []
    for _, article_row in sampled.iterrows():
        scored = score_article_sentences(
            article_row=article_row,
            tokenizer=tokenizer,
            model=model,
            label_map=label_map,
            hypothesis=hypothesis,
            min_sentence_chars=min_sentence_chars,
            min_sentence_words=min_sentence_words,
            batch_size=batch_size,
        )
        if scored.empty:
            continue

        best = scored.iloc[0]
        rows.append(
            {
                'article_id': article_row['id'],
                'title': article_row['title'],
                'best_entailment_prob': best['entailment_prob'],
                'best_claim_margin': best['claim_margin'],
                'best_sentence': best['sentence'],
            }
        )

    result = pd.DataFrame(rows)
    if result.empty:
        return result
    return result.sort_values('best_entailment_prob', ascending=False).reset_index(drop=True)


In [50]:
articles = load_and_clean_guardian_years(
    years=YEARS,
    folder=RAW_DATA_DIR,
    min_body_text_chars=MIN_BODY_TEXT_CHARS,
)

print(f"Loaded {len(articles):,} cleaned Guardian opinion articles from {YEARS}.")
articles[['id', 'date', 'title', 'author_display', 'year']].head(10)


Loaded 61,850 cleaned Guardian opinion articles from [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024].


,id,date,title,author_display,year
0,commentisfree/2015/jan/01/challenge-to-old-order-growing,2015-01-01 09:00:06+00:00,"All over the world, the challenge to the old order is growing",Seumas Milne,2015
1,commentisfree/2015/jan/01/surveillance-2015-everything-play-for,2015-01-01 11:00:08+00:00,"When it comes to surveillance, there is everything to play for",James Ball,2015
2,commentisfree/2015/jan/01/sleep-new-years-resolution-success,2015-01-01 11:30:09+00:00,Sleep your way to new year’s resolution success,Richard Wiseman,2015
3,commentisfree/2015/jan/01/2015-general-election-russell-brand-revolution-young-people-vote,2015-01-01 12:00:09+00:00,If the 2015 general election will be your first – this is your moment,Lucy Webster,2015
4,commentisfree/2015/jan/01/new-year-political-messages,2015-01-01 12:09:23+00:00,New year: the season for grandstanding,Mary Dejevsky,2015
5,commentisfree/oliver-burkeman-column/2015/jan/01/detoxing-debunked,2015-01-01 12:30:10+00:00,'Detoxing' has been debunked. Maybe it's time to debunk that,Oliver Burkeman,2015
6,commentisfree/2015/jan/01/write-about-who-you-are-not-who-you-want-to-become,2015-01-01 12:45:10+00:00,"A letter to myself: Write about who you are, not who you want to become",Mary Pilon,2015
7,commentisfree/2015/jan/01/dear-tony-blair-electorate-shifted-left-ed-miliband,2015-01-01 13:00:03+00:00,"Dear Tony Blair, maybe it’s your fault if the electorate hasn’t shifted to the left",Neal Lawson,2015
8,commentisfree/2015/jan/01/serial-adnan-syed-exonerated-new-trial,2015-01-01 13:25:31+00:00,Will Serial's Adnan Syed ever be exonerated or get a new trial?,Alan Dershowitz,2015
9,commentisfree/2015/jan/01/diet-advice-fat-gp,2015-01-01 14:00:12+00:00,Why not take diet advice from a fat GP? They’re only human,Natalie Haynes,2015


In [51]:
ARTICLE_ID = None
TITLE_QUERY = "palestine"
ARTICLE_INDEX = 0
TOP_N = 10
MIN_SENTENCE_CHARS = 35
MIN_SENTENCE_WORDS = 5
BATCH_SIZE = 16


In [52]:
matching_articles, selected_article = pick_article(
    articles,
    article_id=ARTICLE_ID,
    title_query=TITLE_QUERY,
    article_index=ARTICLE_INDEX,
)

display(matching_articles[['id', 'date', 'title', 'author_display']].head(10))
preview_article(selected_article)


,id,date,title,author_display
0,commentisfree/2015/jan/14/australias-un-vote-on-palestine-does-a-disservice-to-all-sides-including-israelis,2015-01-14 00:34:50+00:00,"Australia's UN vote on Palestine does a disservice to all sides, including Israelis",Bob Carr
1,commentisfree/2015/jun/28/palestine-europe-honour-agreements-israel,2015-06-28 12:16:49+00:00,What Palestine needs is for Europe to honour its own agreements,Saeb Erekat
2,commentisfree/2015/sep/09/british-palestine-sovereign-state-netanyahu-israel,2015-09-09 11:41:56+00:00,The British must formally – and swiftly – recognise Palestine as a sovereign state,"Alon Liel, Ilan Baruch"
3,commentisfree/2016/feb/22/accusations-of-bias-in-coverage-of-the-israel-palestine-conflict,2016-02-22 07:00:54+00:00,Accusations of bias in coverage of the Israel-Palestine conflict,Chris Elliott
4,commentisfree/2016/apr/18/mike-bairds-motherhood-statements-on-palestine-dilute-the-politics,2016-04-18 02:42:32+00:00,Mike Baird's motherhood statements on Palestine dilute the politics,Antony Loewenstein
5,commentisfree/2016/aug/27/why-celtic-fans-flew-flag-for-palestine,2016-08-27 23:05:16+00:00,Why Celtic fans flew the flag for Palestine,Kevin McKenna
6,commentisfree/2016/aug/29/palestine-evictions-human-rights-susiya-bedouin-un-security-council,2016-08-29 15:11:25+00:00,Palestine’s latest evictions are a human rights crisis – world leaders must act,Leilani Farha
7,commentisfree/2016/sep/06/dont-worry-clinton-and-trump-are-going-to-fix-israelpalestine-,2016-09-06 04:00:48+00:00,Don't worry! Clinton and Trump are going to fix Israel/Palestine *,Moustafa Bayoumi
8,commentisfree/2016/sep/21/peace-between-israel-palestine-iran-deal-guide,2016-09-21 19:01:54+00:00,Want peace between Israel and Palestine? The Iran deal is a good guide,Wardah Khalid
9,commentisfree/2017/jan/10/us-embassy-jerusalem-palestine-donald-trump-tel-aviv,2017-01-10 11:44:04+00:00,Moving the US embassy to Jerusalem would destroy Palestine’s hopes of justice,Nur Arafeh


## Australia's UN vote on Palestine does a disservice to all sides, including Israelis

id: commentisfree/2015/jan/14/australias-un-vote-on-palestine-does-a-disservice-to-all-sides-including-israelis
date: 2015-01-14
author(s): Bob Carr

Summary:
Australia’s voting record at the UN on Israeli-Palestinian issues has changed under the Abbott government. A true friend of Israel should be able to send a message about what Australians think

Body preview:
More Australians winced than applauded when they learned of their country’s very last vote at the end of its two year term on the security council. On 29 December Australia was one of only two nations to vote against a Jordanian draft resolution designed to hasten a two-state solution to the Israeli-Palestinian dispute. Once again the conservative side of Australian politics was delivering what the pro-Israel lobby in Australia wanted, ignoring majority Australian opinion and the views of more liberal Jewish Australians. Ignoring as well, the national interests of Australia, the increasingly dire conditions of Palestinians an

In [29]:
tokenizer, model, id2label, label_map = load_nli_model(MODEL_NAME)
print(f"Loaded {MODEL_NAME} on {model_device(model)}.")
print('Model labels:', id2label)
print('Resolved label map:', label_map)


Loaded cross-encoder/nli-deberta-v3-base on mps:0.
Model labels: {0: 'contradiction', 1: 'entailment', 2: 'neutral'}
Resolved label map: {'contradiction': 0, 'neutral': 2, 'entailment': 1}


In [62]:
article=pd.DataFrame(
    [{
        'title':"essay",
        'id': 'essay',
        'body_text': """What is your last dream about? You may already forget it. It is usually hard to
memorize it due to the lack of logic in dreams. Dreams are formed in the collision of random
thoughts and stories in daily life. You never know when or how these pieces can reconstruct
and build a temporal space in your dream, just like forming bonds between chemical
reactants through collisions. Dreams can be created in seconds without notice and fade away
because of their transient nature. Dreams diminish in the flux of thoughts and darkness under
our eyelids at the moment we wake up and continue our lives as usual: a dream is no more
than a brief interlude in our daily lives. However, minor migrants identified as “dreamers”
live in cyclical and endless nightmares, in which they sometimes could not differentiate the
reality and the dream. Their lives follow cycles in two dimensions: temporal and spatial. The
temporal cycle demonstrates how dreamers imagine and anticipate a new life before the
crossing and later struggle with reoccurring memories in the past after they enter the new
country. The spatial cycle illustrates how dreamers first travel to a new country and
eventually yearn for returning home to where all these stories start. While the former parts in
these two cycles are widely publicized under the spotlight, the latter parts are usually ignored.
In the poetry collection Unaccompanied, Javier Zamora, a poet and a dreamer himself,
provides his insights into the re-emergence of trauma in the temporal cycle and yearning for
home in the spatial cycle through blurring boundaries between reality and the dream, the
present and the past, and the new country and homeland.
Dreamers are always identified as minor migrants who are assimilated into the United
States society successfully and can bring value to the country. In the immigration system, the
subgroup of “dreamers” benefits more from the policy than other minor migrants because
they are considered “more deserving” to stay in the United States. At the same time, the
Cen 3
unrealistic expectations derived from the seemingly positive “dreamers” metaphor bring real
and substantial frustrations and stresses to minor migrants. The metaphor of “dreamers”
usually overlooks the influence that minor migrants’ pasts have on their present life.
Looking into stories from undocumented students, it is not hard to find their struggles
with the reappearance of mental trauma. In “Undocumented Students Navigating Academic
Probation and Unrealistic Expectations,” Grecia Monadragón mentions that abuse in
childhood impedes undocumented students at the University of California, Los Angeles
(UCLA), to pursue their bachelor's degree. In Carmen’s case, the unaddressed child abuse
trauma is detrimental to her mental health and affects her academic performance. It is
unrealistic for her to take time off from school and focus to get treatment because “her
undocumented status only placed her in a more precarious status as a nonstudent.”
(Monagragón 51). Similar to Carmen, many other dreamers also suffer from trauma in their
childhood, while violence is also a major reason that drives them to escape from their home
country to a new country. This kind of physical detachment from persecution indeed helps
them protect themselves but still fails to let them escape away from their nightmares. Their
trauma emerges just like a timed bomb under the optimistic “dreamer” metaphor. A piece of
traumatic memory about child abuse in Carmen’s case has the power to drag her back to the
depressing programmed nightmares. The “nonstudent” status echoes “undocumented,”
imposing more pressure on Carmen’s life in the United States. For dreamers, entering a new
country does not mean a fresh start for dreamers’ lives. Instead, they are always trapped in
memories, and the re-emergence of memories eventually creates a cyclical nightmare.
Zamora further develops the idea of reoccurring memories and overlap between the
past and the present. In the poem “Saguaros” in Unaccompanied, Zamora describes his
crossing journey from El Salvador to the United States. More than the shift between
locations, he uses flashbacks to confuse readers with two parallel timelines: dreams about the
Cen 4
crossing in the past and his current life after entering the United States. With the use of
different tenses, Zamora differentiates and connects these two timelines to create a cycle:
immersing in memories of crossing to waking up from the dream and then falling back to the
memory. Not only directly refers to the nightmare by saying that he “wakes up and his throat
is dry,” Zamora responds to the idea of living between reality and dream by monitoring the
tense in the poem. The part between Zamora's “waking and (his) throat is dry” and his
“pulling over” utilizes simple present tense, and parts before and after this section are in the
simple past tense. This difference between tenses suggests that the beginning and end of the
poem is the recollection of the crossing in the past while the middle section is Zamora’s
current life. Although there is a hyphen indicating a break that separates reality and the dream
at the end of the fourth stanza, the shift in tenses is not very obvious and is further mediated
by the actions' wordings. After the hyphen, Zamora follows with “I also scraped needles
first.” The word “Also” shows some intrinsic connections and sequences between his action
of drinking syrup and the action of scraping needles. By emphasizing the shift in tense with a
hyphen and, at the same time, blurring the boundary between reality and dream with a series
of connected actions, Zamora creates a loop in which Zamora is trapped in the absurd
nightmare in which is too realistic that he could not fully separate the reality and the dream.
While some imageries are used in recollections and some in the present time, Zamora
builds inner connections among all of them to create a complete picture and smooth out the
transitions in the loop. Imageries such as “lavender sky,” “red fruits,” “bat,” and “viscous red
syrup” are all linked together to make up a Halloween-like atmosphere. Instead of using
darkness or night as the setting, he chooses dusk when the bats start to come out and
everything is covered with a shallow layer of darkness but not thick enough to mask all their
shape. The lavender color he uses makes the setting dreamy and shows the uniqueness of this
setting, foreshadowing that something strange is about to happen. At that time, this
Cen 5
spectacular scene will be submerged in darkness. The color in the poem becomes more and
more intense followed by “red fruits” and “viscous red syrup,” building up tensions in this
environment. The texture of the syrup is described as “viscous” which also suggests the thick
and sticky liquid is like blood. The association with blood echoes with bats. Bats feed on
blood, but now human is also drinking “blood.” A delicate connection is built between bats
and humans. The poem further increases the intimacy between humans and bats by bringing
in a conversation between bats and humans. A bat said “la sangre del saguaro nos seduce”
(Zamora 13). It seems to be more unrealistic due to the use of another language. In a poem
that is dominated by English, the sentence in Spanish stands out. When readers are unfamiliar
with Spanish, they will be surprised by the sudden change in language. At the first glance,
they may think it is the language of bats. The absurdity builds up step by step to create a
sense of insecurity that transcends time, which is, in some sense, consistent with Carmen’s
trauma which also persists even when she migrates to a new country.
After traveling from his home country to the United States, the spatial loop is
completed by bringing dreamers’ life back to where it begins: their home country. In the
poem “Montage with Mangoes, Volcano, and Flooded Streets,” Zamora recollects his
childhood memories and creates a cyclical and fluid loop not rooted in a fixed time, place, or
subject but more made up of different random pieces. Through blur and incomplete
memories, Zamora incorporates montage styles and flashes of memories that transcend
location to form a hybrid of El Salvador and the United States.
In terms of the technique, montage, a form of art that is composed of fragments of
imagery, creates a scene that does not necessarily have to be rational and explanatory but
expresses emotions in the disorderliness. The montage is a perfect match with the experience
of fragmented memories of child immigrants. This poem is divided into seven stanzas, and
each stanza is a piece of memory with unique characters, locations, and time. “Pluck the
Cen 6
white flor de izote from stems” is about Zamora, and “there’s no way she knew my
technique” is about his mom and dad. “Grab stalk and pull toward belly” is in Salvador, and
“say I can go back” is in the United States. “Put in the bowl to then drop in the pan to mix
with eggs” is in the past, and “climb the big mango tree, branch by branch, up six meters” is
in the future. Stitching these stanzas together to knit them into a flow of emotions, Zamora
writes a poem in a montage style that reflects his true experience as a minor migrant. Far
away from his homeland that he is physically removed from, all around him is unfamiliarity
and unknown. With drastic differences between El Salvador and the United States, recalling
his childhood in his homeland is more challenging. The process of writing about his home
country is like sitting in front of a piece of blank paper and putting anything that comes into
his mind on that paper. These pieces of memories are random, ranging from the natural
landscapes to tastes and smells, but significant in defining his identity and making up his
childhood. These memories are usually fragmental and sometimes even not rational, but it is
this non-linearity that creates this imaginative space that he could return to any time he wants
without a fake passport or risks of being deported. This temporal space is an alternative
reality and a creation with the notion of repetition and circularity.
This poem confuses the setting in terms of distance and location. Zamora uses
volcanoes as an iconic symbol for El Salvador in his childhood memories. Climbing on the
six-meter-high mango trees to see the volcano, Zamora “watches the volcano’s peak fit in
(his) hand.” Zamora grabs the volcano in his hands and wants to stay close to it, expressing
his eagerness to return home. From another perspective, the volcano is huge and it is
impossible to make it fit in a children’s hands. However, since the volcano is far away from
Zamora, it appears to be small and accessible with the notion that a single hand can hold it.
At that time, Zamora ignores the presence of distance and views it as insignificant. However,
when he leaves El Salvador and becomes far away from the volcano, he realizes how the
Cen 7
distance can separate him from his homeland. Even though he knows El Salvador is always
there, and Abuelita is there, now he cannot return to El Salvador. It is not as simple as
touching the volcano’s peak, and the distance is not six meters anymore. He says “I can’t
keep mangoes from falling six meters down.
” at the end of the poem. Yes, he cannot stop
mangoes from falling and splitting open on the ground just like how he cannot avoid his
family splitting up in different countries.
The loop ends and ends at where it starts when Zamora combines El Salvador and the
United States through symbols in the poem. Zamora includes the most representative symbols
of El Salvador in the poem such as mangoes, volcanoes, and Flor de izote. Flor de izote is the
national flower of El Salvador, which connects to Zamora’s yearning for home directly.
Mixing with eggs, the flor de izote emits smells and conveys a taste of hometown, making
the yearning for home more sensational in the multidimensional notion (Zamora 28).
Although imageries selected are mostly iconic of El Salvador, the image of flor de izote also
shadows Zamora’s dual identities. Flor de izote is also planted in parts of California in the
United States. While Zamora mainly utilized this flower as a tie between El Salvador and
himself, he overlooks the embedded duality in this symbol which reflects his transnational
identities. The unawareness of duality can be referred to in the image of “dreamers” who are
assimilated into American society successfully and speak unaccented English. As language is
considered a large part of one’s cultural identity, deprivation of one’s native tongue can be
interpreted as a fundamental denial of that person’s identity. In this poem, the symbol of flor
de izote suggests that Zamora is also slightly influenced by the United States culture and is
unaware of that. The hybrid of two cultures reflects an accurate image of dreamers, and a
combination of El Salvador and the United States is where the spatial loop closes.
The dreamers are constantly trapped in a cyclical nightmare temporally and spatially
starting from fleeing from their home country and ending in yearning to go back. In this
Cen 8
dream, there are absurdity, irrationality, and distortions. Some dreamers may die on their
journey to a new country at the first stage of the loop, and some may live till the time for a
family reunion, but how many of them could escape from this loop?"""
    }]
)

In [63]:
scored_sentences = score_article_sentences(
    article_row=article,
    tokenizer=tokenizer,
    model=model,
    label_map=label_map,
    hypothesis=HYPOTHESIS,
    min_sentence_chars=MIN_SENTENCE_CHARS,
    min_sentence_words=MIN_SENTENCE_WORDS,
    batch_size=BATCH_SIZE,
)

scored_sentences.head(TOP_N)[
    ['sentence_idx', 'entailment_prob', 'neutral_prob', 'contradiction_prob', 'claim_margin', 'sentence']
]


,sentence_idx,entailment_prob,neutral_prob,contradiction_prob,claim_margin,sentence
0,0,0.165205,0.834713,0.000083,0.165122,It is usually hard to\nmemorize it due to the lack of logic in dreams.


In [53]:
scored_sentences = score_article_sentences(
    article_row=selected_article,
    tokenizer=tokenizer,
    model=model,
    label_map=label_map,
    hypothesis=HYPOTHESIS,
    min_sentence_chars=MIN_SENTENCE_CHARS,
    min_sentence_words=MIN_SENTENCE_WORDS,
    batch_size=BATCH_SIZE,
)

scored_sentences.head(TOP_N)[
    ['sentence_idx', 'entailment_prob', 'neutral_prob', 'contradiction_prob', 'claim_margin', 'sentence']
]


,sentence_idx,entailment_prob,neutral_prob,contradiction_prob,claim_margin,sentence
0,29,0.998403,0.001337,0.000260,0.998143,"A poll by Roy Morgan on 5 November, commissioned on behalf of the Australia Palestine Advocacy Network, recorded that 57% of Australians supported a yes vot..."
1,6,0.998290,0.001386,0.000324,0.997966,"It was a lousy way to wrap up a two year term on the security council, given the intense competition to win the seat."
2,75,0.998276,0.001411,0.000314,0.997962,"The whole settlement process has been corrupted, according to the Israeli opposition, with millions in government funds supporting the settlements, run by p..."
3,48,0.998221,0.001437,0.000342,0.997879,By the time the discussion wrapped up the prime minister’s position that Australia should vote no was looking very fragile.
4,50,0.998189,0.001470,0.000341,0.997848,"The next day, at a full meeting of the federal parliamentary Labor party, Gillard faced defeat."
5,45,0.998123,0.001686,0.000190,0.997933,They had lost patience with Israel’s hardline leadership.
6,58,0.998123,0.001659,0.000218,0.997905,"But the Abbott government, elected in September 2013, has proceeded to give the lobby everything."
7,40,0.998111,0.001700,0.000189,0.997921,My position was that Australia should not block the Palestinian bid.
8,87,0.998099,0.001698,0.000203,0.997896,"It’s the only way of bringing the ethno-nationalists in Jerusalem into line, although – subject of course to the current Israeli election – their wrong-head..."
9,41,0.998090,0.001504,0.000405,0.997685,I would have liked a yes vote but was resolved on getting us to abstain at the very least.


In [55]:
sample_top_claims = collect_top_claim_sentence_per_article(
    articles,
    tokenizer=tokenizer,
    model=model,
    label_map=label_map,
    hypothesis=HYPOTHESIS,
    sample_size=100,
    min_sentence_chars=MIN_SENTENCE_CHARS,
    min_sentence_words=MIN_SENTENCE_WORDS,
    batch_size=BATCH_SIZE,
)

sample_top_claims


,article_id,title,best_entailment_prob,best_claim_margin,best_sentence
0,commentisfree/2016/jan/06/corbyn-cabinet-reshuffle-labour,Labour’s reshuffle kerfuffle is over – Corbyn must get on with the real work,0.998488,0.998351,He soon realised that the only way to keep everyone on the party bus was to drive with as much speed as it could bear and make sure no one could get off.
1,commentisfree/article/2024/jul/23/unions-republicans-trump-labor-rights-teamsters-uaw,Unions who think Republicans are warming to labor rights are getting played,0.998481,0.998184,"(Trump offered no explanation why the UAW was responsible for any of this.) Trump then directed his fire at the UAW’s president, Shawn Fain, saying he “shou..."
2,commentisfree/2020/jan/18/fry-up-toast-full-breakfast-death-demise,"The fry-up may be toast – but, much to my own surprise, I don’t really care",0.998466,0.998261,"A recent study for the Grocer found the full breakfast was Britain’s favourite cooked option and, anecdotally, cafe owners report that it is still a bestsel..."
3,commentisfree/2022/may/24/parents-dentists-teeth-grinding-sophie-brickman,We parents are grinding our teeth so much lately that dentists have noticed. Why?,0.998466,0.998298,"As I rock, and shush, and soothe, it dawns on me that all the salves I’m giving him – from the lullabies to the chew toys – will never control the underlyin..."
4,commentisfree/2022/jan/04/with-sydney-hospitals-in-freefall-ive-been-forced-to-make-an-impossible-decision-about-my-sick-elderly-dad,"With Sydney hospitals in freefall, I’ve been forced to make an impossible decision about my sick, elderly dad",0.998461,0.998334,"With visitors now shut out of hospitals and no phones beside patients’ beds, we feared these could be our last moments with Dad."
...,...,...,...,...,...
95,commentisfree/2019/jul/25/birth-certificates-are-currency-for-some-indigenous-australians-their-cost-is-too-high,"Birth certificates are currency. For some Indigenous Australians, their cost is too high",0.998112,0.997868,This means that new parents have to know that they need to do this and request a new form.
96,commentisfree/2018/aug/10/the-guardian-view-on-food-cultures-sharing-not-snatching,"The Guardian view on food cultures: sharing, not snatching",0.998090,0.997982,Activists in the US have targeted the Chicago-based Aloha Poke Co chain (none of its owners understood to be native Hawaiians) for sending legal letters tel...
97,commentisfree/2018/dec/26/the-guardian-view-on-the-restitution-of-cultural-property,The Guardian view on the restitution of cultural property,0.998055,0.997929,"This is not glamorous, nor is it cheap; it is the slow, arduous work of provenance research, and museums must be equipped and resourced to undertake it."
98,commentisfree/2020/aug/03/the-ins-and-outs-of-self-examination,The ins and outs of self-examination,0.997993,0.997791,I was told to return if the problem gets worse.
